# Article Classifier

In [2]:
#imports
import pandas as pd
from tqdm import tqdm
from nltk.tokenize import RegexpTokenizer
import re
from IPython.display import display, HTML

In [3]:
df = pd.read_csv('corp_blogs_parsed.csv', index_col=0)

In [16]:
# rename column 'text ' to 'text'
df = df.rename(columns={'blog_title': 'title'})
df = df.rename(columns={'blog_content': 'body'})

In [17]:
df

,company,title,body
0,abn amro,ABN AMRO biedt bedrijven complete bescherming ...,ABN AMRO biedt midden- en grootbedrijf complet...
1,abn amro,Kunstmatige intelligentie_ Vriend of vijand_ _...,Kunstmatige intelligentie: Vriend of vijand?\n...
2,abn amro,Samen sterk tegen cyberaanvallen van morgen _ ...,Samen sterk tegen cyberaanvallen van morgen\nN...
3,abn amro,Van digitale assistent tot virtuele BFF _ ABN ...,Van digitale assistent tot virtuele BFF\nBlog\...
4,accenture,Beyond the Hype_ Why Agentic AI is Closer Than...,BLOG\nBeyond the hype:\nWhy agentic AI is\nclo...
...,...,...,...
74,tno,How TNO is leading the drive towards sovereign...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
75,tno,Large dataset news organizations for Dutch AI ...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
76,tno,New AI Lab for effective and responsible overs...,Geselecteerde taal: EN\nSustainable\nHealthy\n...
77,ziggo,From Gut Feeling to Data-Driven Decisions_ How...,From Gut Feeling to Data-Driven\nFrom Gut Feel...


In [5]:
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|LLaMA|openai|kunstmatige intelligentie systeem*|intelligente algoritme*|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drone|drones"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord" 
]

def normalize_keyword(k: str) -> str:
    """Convert wildcard-like keywords into safe, precise regex patterns."""

    k = k.strip().lower()

    # algoritme → algoritme, algoritmen, algoritmes
    if k in {"algoritme", "algoritme*"}:
        return r"algoritm(?:e|en|es)"

    # chatbots
    if k in {"chatbot", "chatbot*"}:
        return r"chatbots?"

    # llm / llms
    if k in {"llm", "llm*"}:
        return r"llms?"

    # grote taalmodel / grote taalmodels
    if k in {"grote taalmodel*"}:
        return r"grote taalmodel(?:s)?"

    # intelligent(e) algoritme(n)
    if "intelligente algoritme" in k:
        return r"intelligent[e]?\s+algoritm(?:e|en|es)?"

    # slimme algoritme(n)
    if "slimme algoritme" in k:
        return r"slimm[e]?\s+algoritm(?:e|en|es)?"

    # ANY OTHER keyword with a trailing "*" should become:
    #   <base> → <base>(?:s)?  
    # But only if safe.
    if k.endswith("*"):
        base = k[:-1]
        # optional plural “s”
        return re.escape(base) + r"s?"

    return re.escape(k)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}


In [6]:
_AI_PAT = re.compile(
    r"\b(" + "|".join(normalize_keyword(k) for k in set_ai_words) + r")\b",
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- Remove weiwei articles before classifying ---
def _drop_weiwei_rows(df, title_col='title', body_col='body'):
    mask = (
        df[title_col].astype(str).str.contains(_WEIWEI_PAT, na=False) |
        df[body_col].astype(str).str.contains(_WEIWEI_PAT, na=False)
    )
    removed = mask.sum()
    print(f"Removed {removed} articles containing 'weiwei'.")
    return df.loc[~mask].copy()

_COMPANY_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(n) for n in _COMPANY_NAMES) + r')\b',
    re.IGNORECASE
)
_COMPANY_TOKENS = {n.lower() for n in _COMPANY_NAMES}  # to compare against matched keyword tokens

# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='body'):

    # hard remove weiwei articles
    df = _drop_weiwei_rows(df, title_col, body_col)

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []
    matched_companies_all = [] #store company hits (per row)

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
        
        # --- collect company hits from raw text (title + body)
        comp_title = _COMPANY_PAT.findall(str(title))
        comp_body  = _COMPANY_PAT.findall(str(body))
        company_hits = sorted(set(m.lower() for m in (comp_title + comp_body)))
        matched_companies_all.append(company_hits)

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # ---  ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []
            all_lower = []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        # --- company mention + at least one other keyword -> ai_related = yes ---
        company_present = bool(company_hits)
        other_hits = [m for m in all_lower if m not in _COMPANY_TOKENS]
        if company_present and len(other_hits) >= 1:
            label = "yes"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    df['company_hits'] = matched_companies_all

    return df


In [18]:
# Apply classification (this modifies df and also drops weiwei rows)
df = ai_classification(df)

print(df['ai_related'].value_counts())

Removed 0 articles containing 'weiwei'.


  0%|          | 0/79 [00:00<?, ?it/s]

100%|██████████| 79/79 [00:00<00:00, 280.79it/s]


ai_related
yes    74
no      5
Name: count, dtype: int64


In [19]:
from IPython.display import display, HTML
import pandas as pd
import re

def highlight_keywords(text, keywords):
    """
    Highlights the keywords in the given text with bold, enlarged, and green font.
    'keywords' must be the ACTUAL matched words (e.g. from matched_keywords_all),
    not the raw regex patterns.
    """
    if pd.isna(text):
        return ""
    if not isinstance(keywords, (set, list)):
        raise ValueError("Keywords must be a set or list")
    if len(keywords) == 0:
        return text

    # literal match of each keyword, case-insensitive, with word boundaries
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in keywords) + r')\b'

    def replace_keyword(match):
        kw = match.group(0)
        return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{kw}</span>'

    return re.sub(pattern, replace_keyword, str(text), flags=re.IGNORECASE)


def inspect_ai_related(df, num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords and company names.
    Uses df['matched_keywords_all'] per row instead of the global keyword list.
    """

    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', 'body'}.issubset(df.columns):
        missing = {'title', 'body'} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

    # --- display samples ---
    for idx, row in samples.iterrows():
        ai_val = row['ai_related']
        title = row['title']
        body  = row['body']

        # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x) for x in matched_keywords}
        else:
            # if it's a string or something weird, just wrap it
            row_terms = {str(matched_keywords)} if matched_keywords else set()

        # add company hits if present
        companies = row.get('company_hits', [])
        if isinstance(companies, (list, set, tuple)):
            row_terms.update(str(x).lower() for x in companies)

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if companies:
            header_html += f' &nbsp; | &nbsp; <strong>companies:</strong> {companies}'
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples


In [15]:
df.to_csv('classified_corp_blogs.csv')

In [20]:
df.head()

,company,title,body,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits
0,abn amro,ABN AMRO biedt bedrijven complete bescherming ...,ABN AMRO biedt midden- en grootbedrijf complet...,no,[],[],[],0,0,[facebook]
1,abn amro,Kunstmatige intelligentie_ Vriend of vijand_ _...,Kunstmatige intelligentie: Vriend of vijand?\n...,yes,[],"[ai, algoritmes, kunstmatige intelligentie, ro...","[ai, algoritmes, kunstmatige intelligentie, ro...",0,22,"[apple, facebook, ibm, microsoft]"
2,abn amro,Samen sterk tegen cyberaanvallen van morgen _ ...,Samen sterk tegen cyberaanvallen van morgen\nN...,yes,[],[ai],[ai],0,2,[facebook]
3,abn amro,Van digitale assistent tot virtuele BFF _ ABN ...,Van digitale assistent tot virtuele BFF\nBlog\...,yes,[],"[ai, artificial intelligence, chatbot, chatbot...","[ai, artificial intelligence, chatbot, chatbot...",0,16,"[apple, facebook, microsoft]"
4,accenture,Beyond the Hype_ Why Agentic AI is Closer Than...,BLOG\nBeyond the hype:\nWhy agentic AI is\nclo...,yes,[ai],"[ai, chatgpt, llms]","[ai, chatgpt, llms]",1,59,"[apple, google, nvidia]"


In [21]:
inspect_ai_related(df, num_samples=5, random_state=42)


Displayed 5 random articles (with row-specific matched keywords highlighted).


,company,title,body,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits
30,ing,ING BeleggersBarometer_ Beleggers zien meer ka...,"Zaterdag 29 juni 2024, 08:00 CEST\nBeleggers z...",yes,[generatieve ai],"[ai, generatieve ai, gpt, kunstmatige intellig...","[ai, generatieve ai, gpt, kunstmatige intellig...",1,24,"[adyen, asm international, asml, nvidia]"
0,abn amro,ABN AMRO biedt bedrijven complete bescherming ...,ABN AMRO biedt midden- en grootbedrijf complet...,no,[],[],[],0,0,[facebook]
22,capgemini,Door AI ondersteunde klantinteracties in Neder...,Kunstmatige Intelligentie (AI) is\ndoorgebroke...,yes,[ai],"[ai, chatbots, kunstmatige intelligentie]","[ai, chatbots, kunstmatige intelligentie]",1,58,[]
31,ing,ING introduceert kunstmatige intelligentie-app...,"Woensdag 18 juli 2018, 11:55 CEST\nING introdu...",yes,[kunstmatige intelligentie],"[algoritmen, kunstmatige intelligentie]","[algoritmen, kunstmatige intelligentie]",1,6,[]
18,bosch,#AIoTmatteratBosch _ Bosch in Nederland.pdf,#AIoTMattersAtBosch\nStart video\nDeel dit op...,yes,[],"[ai, algoritme, algoritmes, artificial intelli...","[ai, algoritme, algoritmes, artificial intelli...",0,43,[google]


In [22]:
# Save the classified data
df.to_csv('blogs_classified.csv')

In [ ]:
def sample_articles(df, n=120, random_state=42):
    """
    Return a random sample with columns:
    index, title, body, ai_related, matched_keywords, company_hits
    - Uses 'matched_keywords_all' if present; falls back to 'matched_keywords'
    - If 'company_hits' is missing, fills with empty lists
    """

    # build sample
    k = min(n, len(df))
    samp = df.sample(n=k, random_state=random_state).copy()

    # select/rename columns
    cols = ['title', 'body', 'ai_related', 'company_hits', 'matched_keywords_all']
    samp = samp[cols]

    # keep the original index as a column named 'index'
    samp.insert(0, 'index', samp.index)

    return samp

# usage
sample_df = sample_articles(df, n=120, random_state=42)
sample_df.head()


,index,title,body,ai_related,company_hits,matched_keywords_all
17844,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,no,[],[]
3463,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",no,[],[]
2427,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,no,[],[kunstmatige intelligentie]
3408,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,no,[],[]
7048,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,no,[facebook],[]


In [62]:
# save the sample
sample_df.to_csv('article_sample.csv', index=False)